In [1]:
# Part 3 — Machine Learning: predict no-shows and choose the model honestly

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Load cleaned dataset without re-sorting or shuffling
df = pd.read_csv("appointments_clean.csv")

# Task 3.1 — Features, split, and baseline
feature_cols = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows",
    "reminder_sent",
    "booking_channel",
    "visit_type",
]
X = df[feature_cols].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X columns ({X.shape[1]}): {list(X.columns)}")
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Test no-shows: {int(y_test.sum())}")
print(
    f"Baseline accuracy (always Attended): {(y_test == 0).mean():.4f} "
    f"({int((y_test == 0).sum())} of {len(y_test)})"
)

# Task 3.2 — Logistic regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

print("\n--- Logistic Regression (Test Set, threshold=0.5) ---")
print(f"Accuracy:  {accuracy_score(y_test, lr_pred):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, lr_pred))
print(f"Precision: {precision_score(y_test, lr_pred):.4f}")
print(f"Recall:    {recall_score(y_test, lr_pred):.4f}")
print(f"F1:        {f1_score(y_test, lr_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, lr_prob):.4f}")

print("\nFeature coefficients & scale:")
for col_name, coef_val, scale_val in zip(X.columns, lr.coef_[0], scaler.scale_):
    print(f"  {col_name:26s}: coef={coef_val:+.4f}, scale={scale_val:.4f}")

rem_idx = list(X.columns).index("reminder_sent")
rem_coef = lr.coef_[0][rem_idx]
rem_scale = scaler.scale_[rem_idx]
print(f"\nreminder_sent np.exp(coef):          {np.exp(rem_coef):.4f}")
print(f"reminder_sent np.exp(coef / scale):  {np.exp(rem_coef / rem_scale):.4f}")

# Task 3.3 — Decision tree and overfitting
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
print("\n--- Decision Trees ---")
print(
    f"Full tree (depth={dt_full.get_depth()}): "
    f"train_acc={accuracy_score(y_train, dt_full.predict(X_train)):.4f}, "
    f"test_acc={accuracy_score(y_test, dt_full.predict(X_test)):.4f}"
)

dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)
print(
    f"Depth-3 tree: "
    f"train_acc={accuracy_score(y_train, dt_3.predict(X_train)):.4f}, "
    f"test_acc={accuracy_score(y_test, dt_3.predict(X_test)):.4f}"
)

# Task 3.4 — Random forest and comparison table
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("\n--- Random Forest (n_estimators=100) ---")
print(
    f"train_acc={accuracy_score(y_train, rf.predict(X_train)):.4f}, "
    f"test_acc={accuracy_score(y_test, rf_pred):.4f}, "
    f"recall={recall_score(y_test, rf_pred):.4f}, "
    f"roc_auc={roc_auc_score(y_test, rf_prob):.4f}"
)

top3_imp = sorted(
    zip(X.columns, rf.feature_importances_), key=lambda item: item[1], reverse=True
)[:3]
print("Top 3 feature importances:", [(name, round(float(val), 4)) for name, val in top3_imp])

dt3_prob = dt_3.predict_proba(X_test)[:, 1]
dt3_pred = dt_3.predict(X_test)
comparison_df = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "accuracy": round(accuracy_score(y_test, lr_pred), 4),
            "recall": round(recall_score(y_test, lr_pred), 4),
            "roc_auc": round(roc_auc_score(y_test, lr_prob), 4),
        },
        {
            "model": "Decision Tree (max_depth=3)",
            "accuracy": round(accuracy_score(y_test, dt3_pred), 4),
            "recall": round(recall_score(y_test, dt3_pred), 4),
            "roc_auc": round(roc_auc_score(y_test, dt3_prob), 4),
        },
        {
            "model": "Random Forest (n_estimators=100)",
            "accuracy": round(accuracy_score(y_test, rf_pred), 4),
            "recall": round(recall_score(y_test, rf_pred), 4),
            "roc_auc": round(roc_auc_score(y_test, rf_prob), 4),
        },
    ]
)
print("\nComparison table:\n", comparison_df.to_string(index=False))

# Task 3.5 — Tune, select model & threshold on train CV, then evaluate once on test
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc",
)
grid.fit(X_train, y_train)
print("\n--- GridSearchCV (Train 5-fold CV ROC-AUC) ---")
print("best_params_:", grid.best_params_)
print(f"best_score_:  {grid.best_score_:.4f}")

lr_pipe = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]
)
cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_dt3 = cross_val_score(dt_3, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean()

print("\n5-fold CV ROC-AUC on training set:")
print(f"  Logistic Regression pipeline : {cv_lr:.4f}")
print(f"  Tuned Random Forest          : {grid.best_score_:.4f}")
print(f"  Random Forest (n_estimators=100): {cv_rf:.4f}")
print(f"  Decision Tree (max_depth=3)  : {cv_dt3:.4f}")

oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
print("\nOut-of-fold training threshold comparison:")
for threshold in [0.5, 0.35]:
    oof_flag = (oof >= threshold).astype(int)
    print(
        f"  threshold={threshold:.2f} -> "
        f"precision={precision_score(y_train, oof_flag):.4f}, "
        f"recall={recall_score(y_train, oof_flag):.4f}, "
        f"f1={f1_score(y_train, oof_flag):.4f}, "
        f"flagged={int(oof_flag.sum())}"
    )

chosen_threshold = 0.35
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flags = (probs >= chosen_threshold).astype(int)

print(f"\n--- Final Winner (lr_pipe) on Test Set at threshold={chosen_threshold} ---")
print(f"Precision: {precision_score(y_test, flags):.4f}")
print(f"Recall:    {recall_score(y_test, flags):.4f}")
print(f"Flagged:   {int(flags.sum())}")
print(f"ROC-AUC:   {roc_auc_score(y_test, probs):.4f}")

test_risk_scores = pd.DataFrame(
    {
        "appointment_id": df.loc[X_test.index, "appointment_id"].values,
        "no_show": y_test.values,
        "risk_prob": probs,
        "flag": flags,
    }
)
test_risk_scores.to_csv("test_risk_scores.csv", index=False)

X columns (11): ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train rows: 792, Test rows: 198
Test no-shows: 49
Baseline accuracy (always Attended): 0.7525 (149 of 198)

--- Logistic Regression (Test Set, threshold=0.5) ---
Accuracy:  0.7879
Confusion matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296
Recall:    0.3469
F1:        0.4474
ROC-AUC:   0.7889

Feature coefficients & scale:
  age                       : coef=-0.2539, scale=16.6559
  distance_km               : coef=+0.2534, scale=6.0518
  fee_inr                   : coef=+0.4528, scale=184.2535
  lead_days                 : coef=+0.6145, scale=8.6142
  prior_no_shows            : coef=+0.6496, scale=0.9686
  reminder_sent             : coef=-0.5807, scale=0.4950
  booking_channel_App       : coef=-0.0468, scale=0.4860
  booking_channel_Phone     : coef=+0.2159, scale